# Cross-Sectional Strategies II — Implementation
## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

1. **Combine multiple characteristics** into a composite signal
2. **Recognize when long-short premia are robust vs. fragile**
3. **Estimate transaction-cost impact** on a high-turnover strategy
4. **Recognize crowding effects** when many funds chase the same characteristic
5. **Audit AI-generated cross-sectional code** — z-score winsorization, signal aggregation, turnover

## 📋 TOC
1. [Setup](#setup)  2. [Combining Signals](#combine)
3. [Pitfall Checklist](#pitfalls)  4. [Turnover and Costs](#costs)
5. [Crowding](#crowding)  6. [🎯 Challenge: Robust Multi-Signal Strategy](#challenge)
7. [Submission](#submit)  8. [Key Takeaways](#takeaways)

---
## 🛠️ Setup <a id="setup"></a>

In [ ]:
#@title Setup
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize']=[10,5]; plt.rcParams['font.size']=11
import warnings; warnings.filterwarnings('ignore')
print("✅ Loaded")

---
## Combining Signals <a id="combine"></a>

A single characteristic is rarely the whole story. Real strategies combine
multiple signals — often via simple averaging of z-scores:

$$\text{composite}_{i,t} = \frac{1}{K} \sum_{k=1}^K z(X^{(k)}_{i,t})$$

where $z()$ standardizes each characteristic to have mean 0 and standard
deviation 1 IN THE CROSS-SECTION at time $t$.

Why z-score before averaging? Because raw characteristics have different
units (B/M is a ratio, momentum is a return). Z-scoring puts them on
equal footing.

Why simple averaging? Because **complex weighting schemes are mostly
fitting noise**. The simple average often beats fancier alternatives
out-of-sample.

---
## 🛡️ Pitfall Checklist <a id="pitfalls"></a>

| | Pitfall | What goes wrong | 🔍 How to detect |
|---|---------|-----------------|-------------------|
| 1 | **Z-scoring with the full sample** | Time-$t$ standardization uses time-$t+5$ data | Always z-score within each cross-section, not globally |
| 2 | **Outliers in characteristics** | Single extreme firms can dominate the strategy | Winsorize at 1st/99th percentile before z-scoring |
| 3 | **Missing data treated as zero** | NaN z-score = average | Drop firms with missing characteristics, don't impute zero |
| 4 | **Ignoring transaction costs** | High-turnover strategies look great pre-cost | Always report Sharpe NET of estimated costs |
| 5 | **Equal weighting tiny firms** | Microcap firms dominate equal-weight portfolios | Filter to firms above a minimum market cap (e.g., NYSE 20th percentile) |

---
## Turnover and Costs <a id="costs"></a>

**Turnover** = how much of the portfolio changes from one rebalance to the next.

$$\text{Turnover}_t = \frac{1}{2} \sum_i |w_{i,t} - w_{i,t-1}|$$

A typical value strategy has 30-60% annual turnover. A momentum strategy
has 100-200%.

**Cost** ≈ turnover × spread, where typical liquid-stock spreads are
5-10 bps. A 100% turnover momentum strategy with 10bp costs loses 100 bps/year — meaningful relative to a 4-6% premium.

In [ ]:
# Quick illustration: turnover for two strategies
turnover_value     = 0.40   # 40% per year
turnover_momentum  = 1.50   # 150% per year
spread_bps         = 10     # 10 basis points round-trip

cost_value = turnover_value * spread_bps / 100  # in percent
cost_momentum = turnover_momentum * spread_bps / 100

print(f"Value strategy annual cost:    {cost_value:.2f}%")
print(f"Momentum strategy annual cost: {cost_momentum:.2f}%")
print("\nMomentum costs ~3.75x more than value because of turnover.")

---
## Crowding <a id="crowding"></a>

When many funds chase the same characteristic, the premium shrinks. The
mechanism:
1. Funds enter, push up prices of "value" stocks and push down "growth"
2. The expected mean reversion ($r^{LS}$) shrinks
3. In extreme crowding, the trade reverses temporarily (a "factor crash")

**Quant momentum crashed in August 2007.** Value had a multi-year drought
2010-2019. Both are classic crowding patterns.

> **💡 Why this is hard**
>
> You can rarely observe crowding directly. By the time it shows up in
> performance, you've already taken the loss. The defenses are diversifying
> across many signals, sizing each modestly, and accepting that no signal
> is permanent.

---
## 🎯 Challenge: Robust Multi-Signal Strategy <a id="challenge"></a>

> **Setup.** Suppose three characteristics each predict returns with
> independent Sharpe ratios of 0.4 (modest). Test what happens when you
> combine them.

### Q1 — Combined Sharpe (Pythagoras)

If the three signals are uncorrelated, what's the combined Sharpe?

> **📌 Required:**
> ```python
> sharpe_individual    = 0.4
> n_signals            = 3
> sharpe_combined      = ____   # sqrt(N) * sharpe (when uncorrelated)
> ```

In [ ]:
sharpe_individual = 0.4
n_signals = 3

sharpe_combined = ____
print(f"Combined Sharpe (uncorrelated): {sharpe_combined:.2f}")

### Q2 — Effect of correlation
If signals are 50% correlated instead of uncorrelated, the combined Sharpe
formula changes:

$$SR_{\text{combined}}^2 = \frac{1}{N} \cdot \sum SR_i^2 + \frac{2}{N(N-1)} \sum_{i<j} \rho_{ij} SR_i SR_j$$

Wait — that's wrong. The right formula for an equal-weighted combination is
that the variance of the average signal scales with $(1 + (N-1)\rho)/N$.
For three signals each with Sharpe 0.4 and pairwise correlation 0.5:

$$SR_{\text{combined}} = SR_i \cdot \sqrt{\frac{N}{1 + (N-1)\rho}}$$

> **📌 Required:**
> ```python
> correlation_assumed = 0.5
> sharpe_combined_correlated = ____   # use the formula above
> ```

In [ ]:
correlation_assumed = 0.5
N = n_signals

sharpe_combined_correlated = ____
print(f"Combined Sharpe (50% correlation): {sharpe_combined_correlated:.2f}")
print(f"Loss from correlation: {(1 - sharpe_combined_correlated/sharpe_combined)*100:.1f}%")

### Q3 — Net Sharpe after costs
Your combined strategy has 80% turnover with 10bp costs. Annual cost = 0.8%. If the
gross combined return is 6%/year on a vol of 8%, what's the net Sharpe?

> **📌 Required:**
> ```python
> gross_return = 0.06
> annual_cost  = 0.008
> portfolio_vol = 0.08
> sharpe_net   = ____
> ```

In [ ]:
gross_return = 0.06
annual_cost  = 0.008
portfolio_vol = 0.08

sharpe_net = ____
print(f"Gross Sharpe: {gross_return/portfolio_vol:.2f}")
print(f"Net Sharpe:   {sharpe_net:.2f}")

### Q4 — Memo
Max 5 sentences. Recommend to your CIO whether to deploy a 3-signal cross-sectional
strategy. Cite (i) the combined Sharpe gain, (ii) the correlation cost, (iii) the cost-after-trading.

In [ ]:
MEMO = """Write your memo here."""
print(MEMO)

---
## 📤 Submission <a id="submit"></a>

In [ ]:
# === 📤 SUBMISSION CELL ===
import json, base64, hashlib, datetime as dt
required = ["sharpe_combined", "sharpe_combined_correlated", "sharpe_net", "MEMO"]
missing = [v for v in required if v not in dir()]
if missing: raise NameError(f"\n❌ Missing: {missing}")
payload = {"assignment": "CrossSectional_II_AI",
    "ts": dt.datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "answers": {k: float(eval(k)) for k in required if k != "MEMO"},
    "memo": MEMO.strip()}
blob = json.dumps(payload, sort_keys=True)
token = f"UG54::{hashlib.sha256(blob.encode()).hexdigest()[:8]}::{base64.b64encode(blob.encode()).decode()}"
print("="*72); print(token); print("="*72)

---
## 🧠 Key Takeaways <a id="takeaways"></a>
1. **Combine signals via z-score averaging.** Simple > clever for OOS.
2. **Correlation kills the Pythagoras gain.** Diversification benefits shrink fast as $\rho$ rises.
3. **Always report Sharpe net of costs.** Pre-cost numbers are marketing.
4. **Crowding is real but invisible until it bites.** Size each signal modestly.
5. **AI will hand you a backtest. You have to subtract costs, check correlation, and confirm OOS.**